# Week 2, Lab 4 — Guardrails


In [ ]:
WEEK = 'Week 2'
LAB = 'Lab 4 — guardrails'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn openai openai-agents
else:
    %pip install -q ollama openai openai-agents


In [ ]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


In [ ]:
from agents import input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered

BLOCKLIST = ("password", "api key", "api_key", "credit card")

@input_guardrail
async def no_secrets(ctx, agent, input_data):
    text = input_data if isinstance(input_data, str) else str(input_data)
    tripped = any(w in text.lower() for w in BLOCKLIST)
    return GuardrailFunctionOutput(
        output_info={"text": text, "tripped": tripped},
        tripwire_triggered=tripped,
    )

agent = Agent(
    name="GuardedTutor",
    instructions="Help with the agentic AI course. Be brief.",
    model=model,
    input_guardrails=[no_secrets],
)

async def try_run(prompt: str):
    try:
        result = await Runner.run(agent, prompt)
        print("OK:", result.final_output)
    except InputGuardrailTripwireTriggered:
        print("BLOCKED by input guardrail:", prompt)

await try_run("What is an agent?")
await try_run("Here is my api key sk-test — store it.")


Python guardrails are more reliable than hoping the model will refuse.\n\n**Next:** mini-project.
